In [ ]:
from dataclasses import dataclass
from pathlib import Path
import matplotlib.pyplot as plt
import prism
import warnings
import numpy as np
import time

from imagematerials.concepts import create_electricity_graph
from imagematerials.factory import ModelFactory
from imagematerials.model import GenericStocks, SharesInflowStocks, GenericMaterials, MaterialIntensities
from imagematerials.preprocessing import get_preprocessing_data

from imagematerials.sensitivity_analysis.changedata import change_sector, ChangeAction, ChangeReplace
from imagematerials.sensitivity_analysis.monte_carlo import sample_values, load_material_intensities, load_lifetimes, process_material_intensities

from imagematerials.maintenance import Maintenance
from imagematerials.vehicles.constants import vehicles_modes_sensitivity_analysis
from imagematerials.electricity.constants import EPG_TECHNOLOGIES_FINAL
warnings.filterwarnings("ignore")

knowledge_graph_electricity = create_electricity_graph()

path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials
path_base = Path(path_base, "data", "raw")

In [ ]:
# Get the preprocessing data for the vehicles sector only once
vhc_sector = get_preprocessing_data(
    "vehicles", Path("..", "data", "raw"),
    climate_policy_scenario_dir = Path("..", "data", "raw", "image", "SSP2_baseline"), 
    circular_economy_scenario_dirs = None
)
# sensitivity analysis currently only implemented for road vehicle types (due to lack of data): select those from the preprocessing data
stocks = vhc_sector.all_data["stocks"]
stocks = stocks.sel(Type = vehicles_modes_sensitivity_analysis)
weights = vhc_sector.all_data["weights"]
weights = weights.sel(Type = vehicles_modes_sensitivity_analysis)
mi_maintenance = vhc_sector.all_data["maintenance_material_fractions"]
mi_maintenance = mi_maintenance.sel(Type = vehicles_modes_sensitivity_analysis)
change_definition = {
        "maintenance_material_fractions": ChangeReplace(mi_maintenance),
        "stocks": ChangeReplace(stocks),
        "weights": ChangeReplace(weights),
    }
vhc_sector = change_sector(vhc_sector, change_definition, inplace=True)

In [ ]:
elc_sector = get_preprocessing_data(
    "electricity", Path("..", "data", "raw"),
    climate_policy_scenario_dir = Path("..", "data", "raw", "image", "SSP2_baseline"), 
    circular_economy_scenario_dirs = None
)
# elc_sector is a list of preprocessing data for each electricity subsector

elc_sector_gen = next(item for item in elc_sector if item.name == "elc_gen")

## test MC functions

In [ ]:
ranges = load_material_intensities(path_base / "vehicles" / "vehicles_material_ranges_only_road_vehicles.csv")
rng = np.random.default_rng(42)   # seed once for reproducibility
mi_s = sample_values(ranges, rng=rng)
mi = process_material_intensities(mi_s, "vehicles")

In [ ]:
list(vhc_sector.all_data)

## test model run: generation

In [ ]:
time_start = 2000
time_end = 2055
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

ranges = load_material_intensities(path_base / "electricity" / "standard_data" / "generation_material_intensities_long.csv")
# ranges = ranges.loc[ranges["Cohort"]==2020]
rng = np.random.default_rng(42)   # seed once for reproducibility


start = time.time()
all_output = {}
N = 2
for i in range(N):
    mi = sample_values(ranges, rng=rng)
    mi = process_material_intensities(mi, "electricity")
    
    change_definition = {
        "material_intensities": ChangeReplace(mi),
    }
    new_elc_sector_gen = change_sector(elc_sector_gen, change_definition, inplace=False)
    factory = ModelFactory(
        new_elc_sector_gen, complete_timeline
        ).add(SharesInflowStocks
        ).add(MaterialIntensities
        )
    model = factory.finish()
    model.simulate(simulation_timeline)
    # renaming Type coordinates (necessary due to work around within the sub-technology model for electricity generation)
    rebroadcast_type_dims(model.elc_gen, knowledge_graph_electricity, EPG_TECHNOLOGIES_FINAL)
    all_output[i] = model
    print(f"\rSimulation {i} completed.     ", end="")

end = time.time()
print(f"Total time for {N} simulations: {(end - start)/60:.1f} minutes.")

## test model run: vehicles

In [ ]:
time_start = 2000
time_end = 2055
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

# ranges = load_material_intensities(path_base / "vehicles" / "vehicles_material_ranges.csv")
ranges = load_material_intensities(path_base / "vehicles" / "vehicles_material_ranges_only_road_vehicles.csv")
# ranges = ranges.loc[ranges["Cohort"]==2020]
rng = np.random.default_rng(42)   # seed once for reproducibility


start = time.time()
all_output = {}
N = 2
for i in range(N):
    mi = sample_values(ranges, rng=rng)
    mi = process_material_intensities(mi, "vehicles")
    
    change_definition = {
        "material_fractions": ChangeReplace(mi),
    }
    new_vhc_sector = change_sector(vhc_sector, change_definition, inplace=False)
    factory = ModelFactory(
        new_vhc_sector, complete_timeline
        ).add(GenericStocks
        ).add(GenericMaterials
        )
    model = factory.finish()
    model.simulate(simulation_timeline)
    all_output[i] = model
    print(f"\rSimulation {i} completed.     ", end="")

end = time.time()
print(f"Total time for {N} simulations: {(end - start)/60:.1f} minutes.")

In [ ]:
fig, ax = plt.subplots()
for i in range(N):
    mf = all_output[i].vehicles["material_fractions"]
    plt.plot(mf.Cohort, mf.sel(Type='Cars - ICE', material='aluminium'), label=f"Simulation {i+1}")
plt.xlabel("Time")
plt.ylabel("Material Fractions")
plt.title("Varying Model Input")
# plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
for i in range(N):
    mf = all_output[i].vehicles["inflow_materials"].to_array().sel(time=slice(2005, None)).sum("Region")
    plt.plot(mf.time, mf.sel(Type='Cars - ICE', material='aluminium'), label=f"Simulation {i+1}")
plt.xlabel("Time")
plt.ylabel("Inflow Materials")
plt.title("Varying material demand depending on material intensity")
# plt.legend()
plt.show()